In [7]:
import sys
sys.path.append('..')

In [8]:
import tqdm
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [12]:
def recourse_runner(X: np.ndarray, lar_recourse: LARRecourse, params: dict):
    alpha = params['alpha']
    lamb = params['lamb']
    
    weights_0, bias_0 = lar_recourse.weights, lar_recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    
    X_r = []
    n = len(X)
    for i in tqdm.trange(n, desc=f'Evaluating recourse | alpha={alpha}; lambda={lamb}', colour='#0091ff'):
        x_0 = X[i]
        J = RecourseCost(x_0, lamb)
        
        # Robust Recourse
        x_r = lar_recourse.get_recourse(x_0, beta=1.)
        X_r.append(x_r)
    return np.array(X_r)

In [ ]:
def run_experiment(dataset: Dataset, params: dict, results: List):
    alpha = params['alpha']
    
    for seed in params['seeds']:
        (train_data, test_data) = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        probs = base_model.predict_proba(X_train)[:,-1]
        n_accepted = base_model.predict(X_train).sum()
        print(n_accepted)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        lar_recourse = LARRecourse(weights=weights_0, bias=bias_0, alpha=alpha)
        
        params['lamb'] = lar_recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
        lar_recourse.lamb = params['lamb']
        
        X_r = recourse_runner(recourse_needed_X_train, lar_recourse, params)
        mask = base_model.predict(X_train) == 1
        X_accepted = X_train[mask]
        X_all = np.vstack((X_r, X_accepted))
        y_all = base_model.predict_proba(X_all)
        
        model_p = LR()
        model_p.train(X_all, y_all)
        results.append(model_p)

In [16]:
torch.manual_seed(0)

params = {}
params['alpha'] = 0.5
params['lamb'] = None
params['seeds'] = range(1)
d_results = {}

datasets = [SyntheticDataset()]
for dataset in datasets:
    results = []
    
    print(f'Running {dataset.name} data...')
    run_experiment(dataset, params, results)
    d_results[dataset.name] = results
    
    print(f'Finished {dataset.name}\n')

Running synthetic data...
[[9.99932961e-01 6.70394291e-05]
 [1.08687179e-04 9.99891313e-01]
 [9.97976634e-01 2.02336580e-03]
 ...
 [5.89610384e-05 9.99941039e-01]
 [1.01641612e-02 9.89835839e-01]
 [2.08237350e-05 9.99979176e-01]] 396.0
Choosing lambda


lambda=1.0: 100%|██████████| 404/404 [00:00<00:00, 31544.93it/s]
Evaluating recourse | alpha=0.5; lambda=1.0: 100%|██████████| 404/404 [00:00<00:00, 32773.07it/s]


ValueError: y should be a 1d array, got an array of shape (800, 2) instead.